In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import os

In [2]:
def clean_title(title_html):
    # BeautifulSoup으로 HTML 파싱
    soup = BeautifulSoup(title_html, 'html.parser')
    # mark 태그 제거
    for mark in soup.find_all('mark'):
        mark.unwrap()
    return soup.get_text().strip()

In [3]:
def extract_blog_content(blog_url):
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0'
        }
        res = requests.get(blog_url, headers=headers)
        soup = BeautifulSoup(res.text, 'html.parser')

        # iframe 구조일 경우 본문 실제 주소 추출
        iframe = soup.find('iframe', {'id': 'mainFrame'})
        if iframe:
            real_blog_url = 'https://blog.naver.com' + iframe['src']
            res = requests.get(real_blog_url, headers=headers)
            soup = BeautifulSoup(res.text, 'html.parser')

        # 본문 내용 추출 (클래스는 블로그에 따라 다름, 가장 일반적인 선택자 기준)
        content_area = soup.select_one('div.se-main-container')
        if content_area:
            return content_area.get_text(separator='\n').strip()

        # 구형 블로그 에디터 대응
        content_area = soup.select_one('#postViewArea')
        if content_area:
            return content_area.get_text(separator='\n').strip()

    except Exception as e:
        print(f"본문 크롤링 실패: {blog_url} / 오류: {e}")
        return None

In [23]:
def search_naver_blog(keyword, max_pages=1):
    # 검색 결과를 저장할 리스트
    results = []
    print('시작')
    
    # Selenium 설정
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--window-size=1920,1080')
    
    
    driver = webdriver.Chrome(options=options)
    
    try:
        for page in range(1, 11):
            url = f"https://section.blog.naver.com/Search/Post.naver?pageNo={page}&rangeType=ALL&orderBy=sim&keyword={keyword}"
            print(url)
            driver.get(url)
            
            # 페이지 로딩 대기
            time.sleep(3)
            
            # 블로그 포스트 목록 찾기
           
            posts = driver.find_elements(By.CSS_SELECTOR, "div.area_list_search > div.list_search_post")
            if posts:
                print(f"{len(posts)}개의 포스트 발견!")
            else:
                print('없어용')
            
            for post in posts:
                try:
                   
                    title_elements = post.find_elements(By.CSS_SELECTOR, "div.info_post div.desc a.desc_inner strong")
                    title_element = post.find_element(By.CSS_SELECTOR, "div.info_post div.desc a.desc_inner strong")
                    title = title_element.text.strip()
#                     print('title: ',title)
#                     if title_elements:
#                         print(title_elements)
#                     else:
#                         print('title')
                    
        
                    link_element = post.find_element(By.CSS_SELECTOR, "div.info_post div.desc a.desc_inner")
                    link = link_element.get_attribute("href")
#                     print('link: ',link)

                    try:
                        author_element = post.find_element(By.CSS_SELECTOR, "div.info_post div.writer_info a > em")
                        author_id = author_element.text.strip()
                        print(author_id)

                    except:
                        print('작성자 id 없음: ', link)
                        author_id = "Unknown"
                    
                    # 내용 추출
                    content = extract_blog_content(link)
                    
                    results.append({
                        'author_id': author_id,
                        'title': title,
                        'link': link,
                        'content': content

                    })
                except Exception as e:
                    print(f"게시물 처리 중 오류 발생: {e}")
                    continue
            
            print(f"페이지 {page} 처리 완료")
            print(results)
    
    finally:
        driver.quit()
    
    return result

search_naver_blog('보험설계사')

시작
https://section.blog.naver.com/Search/Post.naver?pageNo=1&rangeType=ALL&orderBy=sim&keyword=보험설계사
7개의 포스트 발견!
title:  KB 태아보험 사은품 다이렉트 산모특약 설계사 추천
link:  https://blog.naver.com/point0453/223884910298
블체사
title:  현대해상태아보험 설계사 추천 내돈내산
link:  https://blog.naver.com/season368/223894550815
수퍼마켓 주인장
title:  현대해상태아보험 내돈내산 설계사 추천!
link:  https://blog.naver.com/1004eihwa/223864444009
새콤달콤
title:  개인회생 보험 설계사 해지 환급금까지 챙겨야죠
link:  https://blog.naver.com/mosu222/223900939117
최홍준 변호사
title:  GA 보험 보험설계사 당신은 이미 오래 참은 겁니다
link:  https://blog.naver.com/somoonga1/223899248537
보험회사
title:  [부업일기] 롯데손해보험 원더 해촉/말소 방법 (손해보험설계사 시험후기, 공부해서...
link:  https://blog.naver.com/tata_blog/223866354683
타타tata
title:  운전자보험 다이렉트 운전자보험 차이알고 설계사 통해 가입해보자
link:  https://blog.naver.com/ktz1118/223894950347
나랑
페이지 1 처리 완료
[{'author_id': '블체사', 'title': 'KB 태아보험 사은품 다이렉트 산모특약 설계사 추천', 'link': 'https://blog.naver.com/point0453/223884910298', 'content': 'KB 태아보험은 첫 아이를 준비하면서 많이 추천받았던 보험 중 하나였어요.\n 주변 엄마들 모임에서 태아보험 얘기 나오면 

7개의 포스트 발견!
title:  실비 암 종합 건강 운전자 외국인보험 설계사 가입 회사 추천 받아 똑부러지게 가입해보기
link:  https://blog.naver.com/chun4112/223873170807
꽃순이아빠
title:  펫보험 인슈랩스 설계사 가입요령 간편하게 살펴보세요
link:  https://blog.naver.com/merry_902/223850080358
Merry
title:  보험설계사연봉 가장 중요한 것은 GA 선택
link:  https://blog.naver.com/green423/223774514428
그린데이
title:  자산관리사, 재무설계사는 보험을 팔지 않을까? 연봉 1억의 진실, 결국 보험설계사.
link:  https://blog.naver.com/e397k09/223895488843
위고
title:  보험설계사 직업 연봉 정년없이 오래할 수 있습니다
link:  https://blog.naver.com/trap4/223885387081
와쿠와쿠
title:  부산 신입 보험설계사 월평균 600만원 벌 수 있는 보험대리점 더금융서비스
link:  https://blog.naver.com/gabiss/223860086909
보험은 여기야
title:  프라임에셋 약관본부, 울산 보험설계사 모집 신입 경력 이직 급여 연봉
link:  https://blog.naver.com/smse0603/223876915858
보험만렙
페이지 2 처리 완료
[{'author_id': '블체사', 'title': 'KB 태아보험 사은품 다이렉트 산모특약 설계사 추천', 'link': 'https://blog.naver.com/point0453/223884910298', 'content': 'KB 태아보험은 첫 아이를 준비하면서 많이 추천받았던 보험 중 하나였어요.\n 주변 엄마들 모임에서 태아보험 얘기 나오면 거의 20~30% 정도는 KB 언급하더라고요. 특히 다이렉트로 가입하면 사은품도 준다고하고, 산모특약도 꽤 다양하다고

7개의 포스트 발견!
title:  보험설계사 설명의무위반 구상금청구소송 방어 사례
link:  https://blog.naver.com/blue5616/223901114070
법무법인 한앤율
title:  자동차 보험 설계사 되는 방법과 수익
link:  https://blog.naver.com/carrgo/223847283047
정비공장 공장장
title:  보험설계사 직업 연봉 급여 걱정없이 버는 조직!!
link:  https://blog.naver.com/rhehrdml10/223867747758
윤성민이사
title:  펫보험 설계사 보장내용 확인해서 알뜰하게 준비하세요
link:  https://blog.naver.com/sooyoung0424/223857231964
널쑤
title:  보험설계사 수수료 구조에 대해 알아보자
link:  https://blog.naver.com/prime6488/223851348472
음본
title:  보험수수료 싹 바뀐다! 지금까지 이런 보험은 없었다, 설계사도 소비자도 달라진다
link:  https://blog.naver.com/bluebelldaum/223887072540
달팽이여유
title:  보험설계사 모집 수당 영업지원 총정리
link:  https://blog.naver.com/somoonga/223741457796
조력자들A
페이지 3 처리 완료
[{'author_id': '블체사', 'title': 'KB 태아보험 사은품 다이렉트 산모특약 설계사 추천', 'link': 'https://blog.naver.com/point0453/223884910298', 'content': 'KB 태아보험은 첫 아이를 준비하면서 많이 추천받았던 보험 중 하나였어요.\n 주변 엄마들 모임에서 태아보험 얘기 나오면 거의 20~30% 정도는 KB 언급하더라고요. 특히 다이렉트로 가입하면 사은품도 준다고하고, 산모특약도 꽤 다양하다고 하니 저도 자연스럽게 관심을 가지게 됐어요. \n\n\n\n\n\n\n\n\n\n\

7개의 포스트 발견!
title:  보험설계사 급여 부업으로도 가능할까?
link:  https://blog.naver.com/kssa7940/223810691262
세일즈랩
title:  보험설계사 자격증 있다면 꼭 알아야합니다.. 교육으로 끝나면 안되는 이유
link:  https://blog.naver.com/dead14449/223873791023
미래전략금융서비스
title:  무자본창업 온라인 DB영업 보험설계사 정착 기간은?
link:  https://blog.naver.com/e397k09/223893090504
위고
title:  동두천 개인회생 보험설계사 카드빚 조건 기간 신청방법 알아보기
link:  https://blog.naver.com/by5831/223870645216
베이비베이비
title:  보험설계사 수수료 수당 전부 확인해드립니다(법인시상, 1인GA)
link:  https://blog.naver.com/wijihoondream/223859100451
베스트 사업단
title:  대전 세종 신입 보험설계사 지인영업NO! 가망고객 무료로 지원하는 보험대리점...
link:  https://blog.naver.com/gabiss/223874429857
보험은 여기야
title:  메리츠 연금보험 보험설계사 수수료 내가 받는 법
link:  https://blog.naver.com/somoontvbiz/223901327064
조력자들R
페이지 4 처리 완료
[{'author_id': '블체사', 'title': 'KB 태아보험 사은품 다이렉트 산모특약 설계사 추천', 'link': 'https://blog.naver.com/point0453/223884910298', 'content': 'KB 태아보험은 첫 아이를 준비하면서 많이 추천받았던 보험 중 하나였어요.\n 주변 엄마들 모임에서 태아보험 얘기 나오면 거의 20~30% 정도는 KB 언급하더라고요. 특히 다이렉트로 가입하면 사은품도 준다고하고, 산모특약도 꽤 다양하다고 하니 저

7개의 포스트 발견!
title:  보험설계사 개인회생, 이렇게 해결할 수 있습니다
link:  https://blog.naver.com/kimjahdsuisui/223865336141
법무법인홍림대표
title:  보험설계사 직업 수당 영업시장 정보 공유
link:  https://blog.naver.com/vkj349wkls/223741493502
보험위키
title:  보험영업 보험설계사를 도전해서 대박난곳!!
link:  https://blog.naver.com/rhehrdml10/223869399012
윤성민이사
title:  보험설계사 급여 3년 연속 증가세 수당 얼마길래?
link:  https://blog.naver.com/vfc0920/223842870245
아이나
title:  보험설계사세금, 3.3%만 떼면 끝난 줄 알았는데요?
link:  https://blog.naver.com/whrbtkd31/223885882898
세무법인 테헤란
title:  보험설계사 급여 온라인영업하면 이정도는 법니다
link:  https://blog.naver.com/pai7923/223879927497
온라인보험
title:  보험판매 수수료 개편_보험설계사 수입 구조 변경? 보도자료 내용 정리
link:  https://blog.naver.com/hwa05093/223887341639
슬금이
페이지 5 처리 완료
[{'author_id': '블체사', 'title': 'KB 태아보험 사은품 다이렉트 산모특약 설계사 추천', 'link': 'https://blog.naver.com/point0453/223884910298', 'content': 'KB 태아보험은 첫 아이를 준비하면서 많이 추천받았던 보험 중 하나였어요.\n 주변 엄마들 모임에서 태아보험 얘기 나오면 거의 20~30% 정도는 KB 언급하더라고요. 특히 다이렉트로 가입하면 사은품도 준다고하고, 산모특약도 꽤 다양하다고 하니 저도 자연스럽게 관심을 가지게 됐어요. \n\n\n\n\n\n\n\n\

...

7개의 포스트 발견!
title:  나에게 맞는 보험설계사 회사찾기 방법을 알아보자
link:  https://blog.naver.com/prime6488/223899573087
음본
title:  돈 잘버는 직업 온라인영업 보험설계사라면 역대연봉이 가능합니다!
link:  https://blog.naver.com/mahapass/223901485609
건달뷰달 마하패스
title:  보험설계사 인식 자격증 시험 난이도 체크
link:  https://blog.naver.com/catconomy_/223902561538
냥코노미
title:  신뢰받는 보험설계사 자격, 손해사정사는 아니더라도
link:  https://blog.naver.com/vfc0920/223880754651
아이나
title:  운전자보험 다이렉트 & 맞춤형 운전자보험 인슈랩스 설계사 통해서 맞춤형으로 가입해보기
link:  https://blog.naver.com/kor4you/223894526219
하얀니
title:  보험설계사 급여 1천만원 이상 벌 수 있는 방법
link:  https://blog.naver.com/pai7923/223858190017
온라인보험
title:  2025년 보험설계사 수수료 개편 임박~!!!
link:  https://blog.naver.com/jean2207/223901150710
진
페이지 7 처리 완료
[{'author_id': '블체사', 'title': 'KB 태아보험 사은품 다이렉트 산모특약 설계사 추천', 'link': 'https://blog.naver.com/point0453/223884910298', 'content': 'KB 태아보험은 첫 아이를 준비하면서 많이 추천받았던 보험 중 하나였어요.\n 주변 엄마들 모임에서 태아보험 얘기 나오면 거의 20~30% 정도는 KB 언급하더라고요. 특히 다이렉트로 가입하면 사은품도 준다고하고, 산모특약도 꽤 다양하다고 하니 저도 자연스럽게 관심을 가지게 됐어요. \n\n\n\n\n\n\n\n

7개의 포스트 발견!
title:  보험설계사 개인회생 소득과 환수금 단순하지는 않죠
link:  https://blog.naver.com/milkplz/223758060838
대표변호사 윤세진
title:  보험설계사 자격증 취득 전 꼭 알아야 하는 현실!
link:  https://blog.naver.com/kssa7940/223896880411
세일즈랩
title:  보험설계사 수당 월급 얼마 받는지 수수료 정보
link:  https://blog.naver.com/jjudds/223871801829
jjudds
title:  경남보험설계사가 알려주는 경남보험 가입 설계 주의사항
link:  https://blog.naver.com/dlswjs1209/223880640254
보험전문가 이선호
title:  30세만기 태아보험 설계사가 추천하는 알찬 구성 설계안 공개합니다
link:  https://blog.naver.com/e397k09/223893008854
위고
title:  보험설계사 월급 얼마나 받을까요?
link:  https://blog.naver.com/wasset1ga/223846199091
더블유에셋
title:  메리츠 저축보험 보험설계사 수수료 내가 받는 법
link:  https://blog.naver.com/somoontvbiz/223902597503
조력자들R
페이지 8 처리 완료
[{'author_id': '블체사', 'title': 'KB 태아보험 사은품 다이렉트 산모특약 설계사 추천', 'link': 'https://blog.naver.com/point0453/223884910298', 'content': 'KB 태아보험은 첫 아이를 준비하면서 많이 추천받았던 보험 중 하나였어요.\n 주변 엄마들 모임에서 태아보험 얘기 나오면 거의 20~30% 정도는 KB 언급하더라고요. 특히 다이렉트로 가입하면 사은품도 준다고하고, 산모특약도 꽤 다양하다고 하니 저도 자연스럽게 관심을 가지게 됐어요. \n\n\n\n\n\n\n\n\n\n\

7개의 포스트 발견!
title:  보험설계사 월급 얼마나 받을까요?
link:  https://blog.naver.com/wasset1ga/223846199091
더블유에셋
title:  보험설계사 온라인으로 고액연봉 달성하는 방법을 알려드립니다!
link:  https://blog.naver.com/mahapass/223881182756
건달뷰달 마하패스
title:  실비 암 종합 건강 운전자 외국인보험 설계사 가입 회사 이용하여 한눈에 비교하고...
link:  https://blog.naver.com/yurublog/223873540087
유루
title:  메리츠 파트너스 부업 보험설계사가 보는 솔직한 의견
link:  https://blog.naver.com/ardorblossom/223897862480
김현준지점장
title:  한화 태아보험 사은품 보험설계사 수수료 받는 법
link:  https://blog.naver.com/somoontvbiz/223870131898
조력자들R
title:  손해보험설계사 자격증 시험 난이도 합격점수 출제경향
link:  https://blog.naver.com/vfc0920/223901100296
아이나
title:  보험설계사 수당 전속 1인GA 비교
link:  https://blog.naver.com/wasset1ga/223821881187
더블유에셋
페이지 9 처리 완료
[{'author_id': '블체사', 'title': 'KB 태아보험 사은품 다이렉트 산모특약 설계사 추천', 'link': 'https://blog.naver.com/point0453/223884910298', 'content': 'KB 태아보험은 첫 아이를 준비하면서 많이 추천받았던 보험 중 하나였어요.\n 주변 엄마들 모임에서 태아보험 얘기 나오면 거의 20~30% 정도는 KB 언급하더라고요. 특히 다이렉트로 가입하면 사은품도 준다고하고, 산모특약도 꽤 다양하다고 하니 저도 자연스럽게 관심을 가지게 됐어요. \n\n\n\n\

7개의 포스트 발견!
title:  1일 자동차보험 설계사가 미리 안내 해야하는 이유
link:  https://blog.naver.com/meg152/223884106044
나인톡
title:  현대해상화재보험 6년차 설계사님 상담 사례
link:  https://blog.naver.com/skirt13688/223903147808
김대표
title:  보험설계사 종합소득세 유의사항과 절세 방법에 대해서
link:  https://blog.naver.com/whrbtkd31/223857726713
세무법인 테헤란
title:  온라인보험설계사 급여 천만원 초보도 가능합니다
link:  https://blog.naver.com/best_pb_han/223863427950
키다리보험아저씨
title:  보험설계사 잘 고르는 방법(feat. 지인은 무조건 걸러라)
link:  https://blog.naver.com/tyugh1004/223875393453
위다드
title:  보험설계사가 되기 위한 자격 요건은?
link:  https://blog.naver.com/prime6488/223826646501
음본
title:  현대해상 태아보험 담보 필수특약/불필요특약 정리 보험비 현대해상 태아보험 설계사...
link:  https://blog.naver.com/liveyourlife_now/223888436073
주디
페이지 10 처리 완료
[{'author_id': '블체사', 'title': 'KB 태아보험 사은품 다이렉트 산모특약 설계사 추천', 'link': 'https://blog.naver.com/point0453/223884910298', 'content': 'KB 태아보험은 첫 아이를 준비하면서 많이 추천받았던 보험 중 하나였어요.\n 주변 엄마들 모임에서 태아보험 얘기 나오면 거의 20~30% 정도는 KB 언급하더라고요. 특히 다이렉트로 가입하면 사은품도 준다고하고, 산모특약도 꽤 다양하다고 하니 저도 자연스럽게 관심을 가지게 됐어요. \n\n

'https://section.blog.naver.com/Search/Post.naver?pageNo=10&rangeType=ALL&orderBy=sim&keyword=보험설계사'

In [24]:
def main():
    # data 디렉토리 생성
    os.makedirs('/app/data', exist_ok=True)
    
    # 검색할 키워드 입력
    keyword = input("검색할 키워드를 입력하세요: ")
    
    # 블로그 검색 실행
    print(f"'{keyword}' 키워드로 네이버 블로그 검색을 시작합니다...")
    results = search_naver_blog(keyword)
    
    # 결과를 DataFrame으로 변환
    df = pd.DataFrame(results)
    
    # 결과 출력
    print("\n검색 결과:")
    # print(df)
    
    # 결과를 CSV 파일로 저장
    filename = f"./data/naver_blog_{keyword}.csv"
    df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"\n결과가 {filename} 파일에 저장되었습니다.")
    print(len(results))
    # print(results[:3])  # 앞에서 3개만 출력


if __name__ == "__main__":
    main() 

검색할 키워드를 입력하세요: 보험설계사 2주 합격
'보험설계사 2주 합격' 키워드로 네이버 블로그 검색을 시작합니다...
시작
https://section.blog.naver.com/Search/Post.naver?pageNo=1&rangeType=ALL&orderBy=sim&keyword=보험설계사 2주 합격
7개의 포스트 발견!
title:  [직장인투잡] 보험설계사 시험 2주 만에 합격한 공부비법!(원더 스마트 플래너 )
link:  https://blog.naver.com/easywayforu/223641816981
이지웨이
title:  보험설계사 시험 2주 준비 / 합격 후기 /투잡성공후기 /손해보험,제3보험,생명보험 시험...
link:  https://blog.naver.com/bblove6042/223744321253
봉봉
title:  2주만에 보험설계사 합격하는 공부법 +25만원
link:  https://blog.naver.com/cherishyy_/223715753626
cherish
title:  손해보험 설계사 시험 합격과정
link:  https://blog.naver.com/bobba_/223881156811
깝녕
title:  손해보험 설계사 자격시험 보려면? 절차·과목·합격 기준
link:  https://blog.naver.com/gurugram/223892159706
야간열차
title:  창원 마산 생명보험설계사 시험 응시방법 및 합격조회
link:  https://blog.naver.com/leehw1322/223900714731
leehw1322
title:  [2025.3] DB손해보험 설계사 자격시험 합격 후기 (시험장:광주)
link:  https://blog.naver.com/alfks6194/223810879005
하루
페이지 1 처리 완료
[{'author_id': '이지웨이', 'title': '[직장인투잡] 보험설계사 시험 2주 만에 합격한 공부비법!(원더 스마트 플래너 )', 'link': 'https

7개의 포스트 발견!
title:  억대 연봉 보험설계사 시험 99.9% 합격하는 준비방법
link:  https://blog.naver.com/future-via-garden/223871681904
금융전문가 WON
title:  변액보험설계사 시험, 한화생명 VIP지점장이 알려주는 단기 합격 전략
link:  https://blog.naver.com/totov4739/223857513705
한화생명금융서비스
title:  24년 7월 손해보험설계사 시험 합격 후기 (ft. 서울 시험장 한국기독교연합회관 6층)
link:  https://blog.naver.com/jinin_4/223527906761
진인사w
title:  2025 보험설계사 자격증 시험 최신 난이도 분석과 합격 전략 꿀팁
link:  https://blog.naver.com/hybum1/223868867251
날고싶은커피향
title:  손해보험설계사 자격시험 공부 과정, 시험 고득점 합격 후기
link:  https://blog.naver.com/hchh1820/223498121657
하하맘
title:  대구 프라임에셋 손해보험설계사 시험응시 방법 및 합격방법
link:  https://blog.naver.com/prime8721/223871415533
손정락 부지점장
title:  생명보험설계사 자격시험 합격 후기(생명보험-93점, 제3보험-97점)
link:  https://blog.naver.com/cartmanlove/223638114781
ㅇㅇ
페이지 2 처리 완료
[{'author_id': '이지웨이', 'title': '[직장인투잡] 보험설계사 시험 2주 만에 합격한 공부비법!(원더 스마트 플래너 )', 'link': 'https://blog.naver.com/easywayforu/223641816981', 'content': '저는 \n"원더 스마트 플래너 앱"\n으로 투잡을 시작한 직장인입니다.\n10월에 처음 원더앱을 알게 되고, 바로 가입 후 현재는 보험설계사 위촉도

7개의 포스트 발견!
title:  손해보험설계사 시험 합격 후기 난이도 확인
link:  https://blog.naver.com/dngnsrn78362/223825138523
우니
title:  누구나 보험설계사 합격하고 25만원 받는 방법!
link:  https://blog.naver.com/cherishyy_/223678595359
cherish
title:  손해보험설계사 시험 접수방법, 공부방법, 합격조회까지
link:  https://blog.naver.com/10041004ing/223835164073
AZ
title:  어짜다보니 ....보험 설계사 FP시험합격～
link:  https://blog.naver.com/dahai_/223884854659
dahai_
title:  손해보험설계사 시험 합격후기 :) 공부 방법
link:  https://blog.naver.com/forever_eun_/223351434850
은냥이
title:  보험설계사 신입 시험 / 자격증 준비부터 합격 후 절차까지 한 번에 정리!!(신입설계사)
link:  https://blog.naver.com/js6545/223836013902
아너김진수지점장
title:  손해보험 설계사 시험 합격 후기 설계사 시험 위치 시험 결과 조회 첫 보험 계약 청약
link:  https://blog.naver.com/ur2754/223280471442
우룩
페이지 3 처리 완료
[{'author_id': '이지웨이', 'title': '[직장인투잡] 보험설계사 시험 2주 만에 합격한 공부비법!(원더 스마트 플래너 )', 'link': 'https://blog.naver.com/easywayforu/223641816981', 'content': '저는 \n"원더 스마트 플래너 앱"\n으로 투잡을 시작한 직장인입니다.\n10월에 처음 원더앱을 알게 되고, 바로 가입 후 현재는 보험설계사 위촉도 마친 상태에요~!\n\u200b\n원더앱은 롯데손해보험에서 운영하는 앱이고, \n원더앱

7개의 포스트 발견!
title:  *3월 이후 종료* 원더 보험설계사 합격하고 25만원 + 제 추천으로 125000원 받아가세요...
link:  https://blog.naver.com/irene567/223712601857
달경
title:  생명보험협회 자격시험 / FP자격시험 보험설계사시험 2일 공부하고 합격한 썰 : 1편
link:  https://blog.naver.com/gohititgo/223355375788
오로지
title:  손해보험/제3보험 설계사 합격 후기
link:  https://blog.naver.com/doctorabbit/223620324623
죽순의 목이버섯
title:  보험 설계사 합격하면 25만원을 준다고? 믿을 수 없어서 직접 해보는 중 (feat. 원더...
link:  https://blog.naver.com/mirankim123/223694207340
사요미
title:  내가 보험설계사 자격증을 딴 이유(보험 자격증 종류)
link:  https://blog.naver.com/doomaker/223870105489
모아이형
title:  24/01/12(금) | 손해보험 설계사 자격시험 합격
link:  https://blog.naver.com/jjuya0124/223340005380
김현주 세무사
title:  보험설계사 첫달
link:  https://blog.naver.com/realstalker/223576342322
아름다운화
페이지 4 처리 완료
[{'author_id': '이지웨이', 'title': '[직장인투잡] 보험설계사 시험 2주 만에 합격한 공부비법!(원더 스마트 플래너 )', 'link': 'https://blog.naver.com/easywayforu/223641816981', 'content': '저는 \n"원더 스마트 플래너 앱"\n으로 투잡을 시작한 직장인입니다.\n10월에 처음 원더앱을 알게 되고, 바로 가입 후 현재는 보험설계사 위촉도 마친 상태에요~!\n\u200b\n원더앱은

7개의 포스트 발견!
title:  새로운 도전, 보험설계사 자격증 시험 도전
link:  https://blog.naver.com/bbirihi/223877955729
소소린
title:  보험 부업 엔잡 추가 그것도 비대면으로 설계사 모집 중
link:  https://blog.naver.com/moneylightning/223902052786
머니 라이트닝 머라
title:  [전남] DB손해보험 설계사 FTC 교육과정 찐~후기 (지인영업 X)
link:  https://blog.naver.com/alfks6194/223816996660
하루
title:  보험설계사 월급통장 얼마? 급여명세서 공개! 취직취업 100%
link:  https://blog.naver.com/bloom10004/223796148832
박버드의 노빠꾸보험
title:  직접 내 보험 설계하려고 보험설계사 됨 보험설계사 시험 후기 합격 30세 미만 어린이보험
link:  https://blog.naver.com/ur2754/223238443839
우룩
title:  [부업일기] 롯데손해보험 원더 해촉/말소 방법 (손해보험설계사 시험후기, 공부해서...
link:  https://blog.naver.com/tata_blog/223866354683
타타tata
title:  대전 세종 신입 보험설계사 지인영업NO! 가망고객 무료로 지원하는 보험대리점...
link:  https://blog.naver.com/gabiss/223874429857
보험은 여기야
페이지 5 처리 완료
[{'author_id': '이지웨이', 'title': '[직장인투잡] 보험설계사 시험 2주 만에 합격한 공부비법!(원더 스마트 플래너 )', 'link': 'https://blog.naver.com/easywayforu/223641816981', 'content': '저는 \n"원더 스마트 플래너 앱"\n으로 투잡을 시작한 직장인입니다.\n10월에 처음 원더앱을 알게 되고, 바로 가입 후 현재는 보험설계사

...

7개의 포스트 발견!
title:  보험설계사 자격증 초단기 준비방법
link:  https://blog.naver.com/jdsa0101/223828226155
인카제이어스성남본부
title:  손해설계사 자격시험 공부법, 단기내 합격 필승 전략 !
link:  https://blog.naver.com/hanwhains_suwon/223780558798
강의하는 심팀장
title:  손해보험설계사 준비한다면? 자격시험 준비방법 (목포DB손해보험 / BTC 과정)
link:  https://blog.naver.com/alfks6194/223808177361
하루
title:  생명보험 설계사 시험
link:  https://blog.naver.com/jason_kim00/223853621201
jason_kim00
title:  부산 신입 보험설계사 월평균 600만원 벌 수 있는 보험대리점 더금융서비스
link:  https://blog.naver.com/gabiss/223860086909
보험은 여기야
title:  천안 주부/경단녀 일자리 보험설계사로 월700만원 이상 버는 보험대리점 더금융서비스
link:  https://blog.naver.com/morffstyle/223867959824
morffstyle
title:  마산 창원 손해보험설계사 시험 응시방법과 공부방법
link:  https://blog.naver.com/leehw1322/223837039685
leehw1322
페이지 7 처리 완료
[{'author_id': '이지웨이', 'title': '[직장인투잡] 보험설계사 시험 2주 만에 합격한 공부비법!(원더 스마트 플래너 )', 'link': 'https://blog.naver.com/easywayforu/223641816981', 'content': '저는 \n"원더 스마트 플래너 앱"\n으로 투잡을 시작한 직장인입니다.\n10월에 처음 원더앱을 알게 되고, 바로 가입 후 현재는 보험설계사 위촉도 마친 상태에요~!\n\u200b\n원더

7개의 포스트 발견!
title:  보험설계사 되는 법 – 시작이 반입니다!
link:  https://blog.naver.com/sj20653/223826557187
보험설계사 김상진
title:  #안산 #시흥 #화성 교보생명 "전속" 보험설계사 모집
link:  https://blog.naver.com/kb_kimriwon/223790707681
교보생명 김리원FP
title:  손해보험설계사 시험 결과 빠르게 확인하는 법
link:  https://blog.naver.com/oljnd2331/223817068270
gaia20006
title:  라이프엠디 LIFE MD 보험설계사 준비 집에서 앱으로 간단하게 도전해보자!
link:  https://blog.naver.com/yel_1004/222288474854
소국
title:  2025년 손해보험설계사 시험 준비법, 최신 경향까지 총정리
link:  https://blog.naver.com/totov4739/223845746307
한화생명금융서비스
title:  보험설계사 자격증 이대로만 준비하세요
link:  https://blog.naver.com/entks1016/223790997532
백두산 보험
title:  손해보험설계사 시험 교육 시작
link:  https://blog.naver.com/exxelhabit/223796548030
보험설계사 권태훈
페이지 8 처리 완료
[{'author_id': '이지웨이', 'title': '[직장인투잡] 보험설계사 시험 2주 만에 합격한 공부비법!(원더 스마트 플래너 )', 'link': 'https://blog.naver.com/easywayforu/223641816981', 'content': '저는 \n"원더 스마트 플래너 앱"\n으로 투잡을 시작한 직장인입니다.\n10월에 처음 원더앱을 알게 되고, 바로 가입 후 현재는 보험설계사 위촉도 마친 상태에요~!\n\u200b\n원더앱은 롯데손해보험에서 운영하는 앱이고, \n원더앱 가입을 하고,  보

7개의 포스트 발견!
title:  수원 보험 설계사 도전기! 보험설계사 자격증 따는 법!
link:  https://blog.naver.com/limcy0928/223589244835
하하맘
title:  직장인 N잡, 롯데손해보험 원더앱으로 보험설계사 되기!
link:  https://blog.naver.com/wons0_0/223729116305
밝음
title:  손해보험 설계사 시험 보고온 후기
link:  https://blog.naver.com/rohyeonfamily/223396364217
로둥마미
title:  천안 아산 더금융서비스 주부/경단녀 보험설계사가 월평균 600만원 벌 수 있는...
link:  https://blog.naver.com/morffstyle/223844904669
morffstyle
title:  AFPK 자격증 2주 독학 합격 후기, 공부방법 +시험장, 준비물 팁
link:  https://blog.naver.com/fnhelp/223778279959
해커스금융
title:  AZ 보험대리점 GA에서 알려주는 손해보험설계사 준비
link:  https://blog.naver.com/ehakdwk9003/223896626536
권시우 본부장
title:  88회 AFPK 대학생 합격후기, 금융권 취업 준비생+공부방법 #재무설계사
link:  https://blog.naver.com/tomato_tv/223613801604
토마토증권통
페이지 9 처리 완료
[{'author_id': '이지웨이', 'title': '[직장인투잡] 보험설계사 시험 2주 만에 합격한 공부비법!(원더 스마트 플래너 )', 'link': 'https://blog.naver.com/easywayforu/223641816981', 'content': '저는 \n"원더 스마트 플래너 앱"\n으로 투잡을 시작한 직장인입니다.\n10월에 처음 원더앱을 알게 되고, 바로 가입 후 현재는 보험설계사 위촉도 마친 상태에요~!\n\u200b\n원더앱은 롯데손해보

7개의 포스트 발견!
title:  천안 보험대리점 GA 변액보험설계사 자격증 따는 법과 중요성
link:  https://blog.naver.com/yfs724/223896584067
유퍼스트 정지혜팀장
title:  내 보험 영리하게 가입하고 소득 얻을 수 있는 롯데손해보험 원더(wonder)
link:  https://blog.naver.com/lth0713/223900383079
영리한 오스카
title:  변액보험설계사 시험, 왜 중요한가요? 에즈금융서비스가 설명합니다
link:  https://blog.naver.com/10041004ing/223887959012
AZ
title:  안녕하세요! 메리츠화재 보험설계사 김세진 인사 올려요!
link:  https://blog.naver.com/meritz84/223890730606
메리츠화재 설계사
title:  초보자도 가능한 손해보험설계사 준비법! 대구 TC사업단에서 알려드립니다
link:  https://blog.naver.com/murne1944/223874984724
KB손해보험정혜수팀장
title:  대구 보험대리점 프라임에셋 변액보험설계사 응시방법 및 중요성
link:  https://blog.naver.com/prime8721/223886932361
손정락 부지점장
title:  집에서 하는 부업 엔잡러에게 딱인 롯데손해보험 원더 (wonder)
link:  https://blog.naver.com/bling-777/223900739073
블링봉봉
페이지 10 처리 완료
[{'author_id': '이지웨이', 'title': '[직장인투잡] 보험설계사 시험 2주 만에 합격한 공부비법!(원더 스마트 플래너 )', 'link': 'https://blog.naver.com/easywayforu/223641816981', 'content': '저는 \n"원더 스마트 플래너 앱"\n으로 투잡을 시작한 직장인입니다.\n10월에 처음 원더앱을 알게 되고, 바로 가입 후 현재는 보험설계사 위촉도 마친

ValueError: DataFrame constructor not properly called!